In [1]:
import os
import mne
import numpy as np
import pandas as pd
from scipy import interpolate
import matplotlib.pyplot as plt
import scipy
import sklearn
from scipy.signal import resample
import json
import warnings
from collections import defaultdict
warnings.filterwarnings("ignore")

In [2]:
import mne
from mne.preprocessing import ICA
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [3]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 200  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 100    # e.g. 200 means 50% overlap for SEG_LEN=400

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L200'

In [4]:
# root dir
root = 'BACA-RS/'
# participants file path
participants_path = os.path.join(root, 'participants.tsv')
participants = pd.read_csv(participants_path, sep='\t')
participants

,participant_id,sex,age,handedness,session1,late_ses1,session2,late_ses2
0,sub-001,F,60,right,yes,0,yes,0.0
1,sub-002,M,67,right,yes,0,no,NaN
2,sub-003,F,44,right,yes,0,no,NaN
3,sub-004,F,24,right,yes,0,no,NaN
4,sub-005,F,48,right,yes,0,yes,5.0
...,...,...,...,...,...,...,...,...
603,sub-604,M,29,right,yes,0,no,NaN
604,sub-605,M,51,right,yes,0,no,NaN
605,sub-606,F,61,right,yes,0,yes,0.0
606,sub-607,M,66,right,yes,0,no,NaN


In [5]:
# Test for bad channels, sampling freq and shape
bad_channel_list, sampling_freq_list, data_shape_list = [], [], []
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub)
        for session in os.listdir(sub_path): 
            if 'sessions' in session:  # skip json and tsv files
                continue
            session_path = os.path.join(sub_path, session, 'eeg/')
            print(session_path)
            for file in os.listdir(session_path):
                if '.edf' in file:
                    file_path = os.path.join(session_path, file)
                    raw = mne.io.read_raw_edf(file_path, preload=True)
                    # print(raw.get_montage())
                    # get bad channels
                    bad_channel = raw.info['bads']
                    bad_channel_list.append(bad_channel)
                    # get sampling frequency
                    sampling_freq = raw.info['sfreq']
                    sampling_freq_list.append(sampling_freq)
                    # get eeg data
                    data = raw.get_data()
                    data_shape = data.shape
                    data_shape_list.append(data_shape)
# Subject 29 and 104 cannot be loaded

BACA-RS/sub-001\ses-1\eeg/
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-TMLR-Rebuttal\BACA-RS\BACA-RS\sub-001\ses-1\eeg\sub-001_ses-1_task-EyesClosed_acq-post_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 192999  =      0.000 ...   192.999 secs...
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-TMLR-Rebuttal\BACA-RS\BACA-RS\sub-001\ses-1\eeg\sub-001_ses-1_task-EyesClosed_acq-pre_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 183999  =      0.000 ...   183.999 secs...
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-TMLR-Rebuttal\BACA-RS\BACA-RS\sub-001\ses-1\eeg\sub-001_ses-1_task-EyesOpen_acq-post_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 184999  =      0.000 ...   184.999 secs...
Extracting EDF parameters

In [6]:
from collections import Counter

print(bad_channel_list)
print(data_shape_list[0])
print("Channel number counter:", Counter(i[0] for i in data_shape_list))
print("Sampling rate counter:", Counter(sampling_freq_list))

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [],

In [7]:
common_channels = None
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub)
        for ses in os.listdir(sub_path):
            if 'ses-' in ses:
                eeg_path = os.path.join(root, sub, ses, 'eeg/')
                for file in os.listdir(eeg_path):
                    if file.endswith('.edf'):
                        raw_tmp = mne.io.read_raw_edf(os.path.join(eeg_path, file), preload=False)
                        chans = set(raw_tmp.ch_names)
                        if common_channels is None:
                            common_channels = chans
                        else:
                            common_channels &= chans

common_channels = list(common_channels) if common_channels else []
print("Final common channels:", common_channels)

Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-TMLR-Rebuttal\BACA-RS\BACA-RS\sub-001\ses-1\eeg\sub-001_ses-1_task-EyesClosed_acq-post_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-TMLR-Rebuttal\BACA-RS\BACA-RS\sub-001\ses-1\eeg\sub-001_ses-1_task-EyesClosed_acq-pre_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-TMLR-Rebuttal\BACA-RS\BACA-RS\sub-001\ses-1\eeg\sub-001_ses-1_task-EyesOpen_acq-post_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-TMLR-Rebuttal\BACA-RS\BACA-RS\sub-001\ses-1\eeg\sub-001_ses-1_task-EyesOpen_acq-pre_eeg.edf...
EDF file detected
Setting channel info structure.

In [8]:
# remove non-eeg electrodes
non_eeg_channels = [
    'Status', 
]
common_channels = [ch for ch in common_channels if ch not in non_eeg_channels]
print(common_channels)
print("Common channels number: ", len(common_channels))

['FC1', 'F4', 'FC6', 'FC3', 'AF8', 'PO8', 'AF3', 'FT10', 'C5', 'F5', 'AF4', 'CP2', 'F7', 'T8', 'Pz', 'CPz', 'PO3', 'P4', 'F1', 'F2', 'POz', 'CP5', 'F8', 'T7', 'Cz', 'AF7', 'C2', 'TP9', 'TP7', 'CP3', 'TP8', 'Fz', 'P8', 'P5', 'C6', 'C4', 'FC2', 'Fp1', 'FT7', 'P7', 'CP1', 'F6', 'Oz', 'C3', 'PO9', 'P2', 'O2', 'FT9', 'PO4', 'C1', 'P6', 'TP10', 'Fp2', 'CP6', 'FC4', 'CP4', 'P3', 'P1', 'PO7', 'PO10', 'O1', 'FC5', 'F3', 'FT8']
Common channels number:  64


## Features

In [9]:
def data_preprocessing(
    raw: mne.io.Raw,
    notch_freq: float = 60.0,
    l_freq: float = 0.5,
    h_freq: float = 40.0,
    do_bad_interp: bool = True,
    verbose: bool = True,
):
    """
    Preprocessing steps ：
      2) Set Montage 
      3) 60 Hz Notch（before band pass）
      4) bandpass filter（default 0.5–40 Hz）
      5) interpolate bad channels（if do_bad_interp is True）
      6) re-reference to average
      7) ICA（在 1 Hz 高通的副本上拟合，自动剔除眼动/肌电等分量，需 mne-icalabel）
    """
        
    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1005'))
    if verbose:
        print("✔ Step 2, Montage set: 'standard_1005'.")
        
    # 3. Notch（工频）
    if notch_freq is not None:
        raw.notch_filter(freqs=[notch_freq], picks="eeg", verbose=False)
        if verbose:
            print(f"✔ Step 3: Notch @ {notch_freq} Hz")
        
    # 4. Bandpass Filter (0.5–40 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Step 4: Band-pass {l_freq}–{h_freq} Hz")
        
    # 5. Interpolate bad channels
    if do_bad_interp and raw.info.get("bads"):
        raw.interpolate_bads(reset_bads=True, verbose=False)
        if verbose:
            print(f"✔ Step 5: Interpolated bads: {raw.info.get('bads', [])}")
    else:
        if verbose:
            print("ℹ Step 5: No bads to interpolate (set raw.info['bads'] first if needed)")
            
    # 6) Re-reference to average
    raw.set_eeg_reference("average", verbose=False)
    if verbose:
        print("✔ Step 6: Average reference")
    
    # 5. ICA + ICLabel
    raw_ica = raw.copy()
    ica = ICA(n_components=None, random_state=97, max_iter='auto')
    ica.fit(raw_ica)
    if verbose:
        print("✔ ICA fitted.")

    try:
        ic_labels = label_components(raw_ica, ica, method='iclabel')
        labels = ic_labels['labels']
        probs = ic_labels['y_pred_proba']

        artifact_thresholds = {
            'eye blink': 0.7,
            'muscle artifact': 0.6,
            'heart beat': 0.5,
            'line noise': 0.8,
            'channel noise': 0.9
        }

        to_exclude = [
            i for i, label in enumerate(labels)
            if label in artifact_thresholds and probs[i] >= artifact_thresholds[label]
        ]
        if to_exclude:
            ica.exclude = to_exclude
            ica.apply(raw)
            if verbose:
                print(f"✔ ICA applied. Excluded components: {to_exclude}")
        else:
            if verbose:
                print("No ICs exceeded artifact thresholds. No components excluded.")

    except Exception as e:
        if verbose:
            print(f"⚠ ICLabel failed: {e}. Proceeding without ICA-based removal.")
        
    return raw

In [10]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [11]:
# Loop through each subject and session to preprocess the EEG data
session_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
standard_channels = common_channels
n_channels = len(standard_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP, "\n")

ses_id = 1   # as two session has 5 years gap, we treat them as different subjects
for sub in os.listdir(root):
    if 'sub-' not in sub:
        continue
    else:
        sub_path = os.path.join(root, sub)
        for session in os.listdir(sub_path): 
            if 'sessions' in session:  # skip json and tsv files
                continue
            session_path = os.path.join(sub_path, session, 'eeg/')
            print(session_path)
            for file in os.listdir(session_path):  # loop through eye open/closed files
                if '.edf' in file:
                    file_path = os.path.join(session_path, file)
                    raw = mne.io.read_raw_edf(file_path, preload=True)
                    original_fs = raw.info['sfreq']
                    T_raw = raw.n_times
                    print("  Original length T_raw =", T_raw)
                    for fs in SAMPLE_RATE_LIST:
                        T_res = int(T_raw * fs / original_fs)
                        # compute number of segments
                        n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                        session_segment_counts[ses_id][fs] += n_seg
                        N_total += n_seg
                        print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
                        print("----------------------------------------")
            print("Total segments for Session", ses_id, ":", sum(session_segment_counts[ses_id][fs] for fs in SAMPLE_RATE_LIST))
            ses_id += 1
            print("----------------End of Session-----------------\n")
    print("----------------End of Subject-----------------\n")

SEG_LEN = 200 OVERLAP = 100 STEP = 100 

BACA-RS/sub-001\ses-1\eeg/
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-TMLR-Rebuttal\BACA-RS\BACA-RS\sub-001\ses-1\eeg\sub-001_ses-1_task-EyesClosed_acq-post_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 192999  =      0.000 ...   192.999 secs...
  Original length T_raw = 193000
fs=200Hz: T_res=38600, STEP=100, n_seg=385
----------------------------------------
fs=100Hz: T_res=19300, STEP=100, n_seg=192
----------------------------------------
fs=50Hz: T_res=9650, STEP=100, n_seg=95
----------------------------------------
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-TMLR-Rebuttal\BACA-RS\BACA-RS\sub-001\ses-1\eeg\sub-001_ses-1_task-EyesClosed_acq-pre_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 183999  =      0.000 ...   183.999 secs...
  Original length

In [12]:
output_root = os.path.join('Processed', sub_folder_path, 'BACA-RS')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))
print("Created memmap for X with shape:", X_mm.shape)

X path: Processed\L200\BACA-RS\X.dat
y path: Processed\L200\BACA-RS\y.dat
Created memmap for X with shape: (2128812, 200, 64)


## PASS 2: Process and store segments into memmap

In [13]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation
ses_id = 1   # as two session has 5 years gap, we treat them as different subjects
for sub in os.listdir(root):
    if 'sub-' not in sub:
        continue
    else:
        sub_path = os.path.join(root, sub)
        for session in os.listdir(sub_path): 
            if 'sessions' in session:  # skip json and tsv files
                continue
            label = 0 # all subjects are healthy controls
            session_path = os.path.join(sub_path, session, 'eeg/')
            print(session_path)
            total_seg_ses = sum(session_segment_counts[ses_id][fs] for fs in SAMPLE_RATE_LIST)
            if total_seg_ses == 0:
                continue
            print(f"[PASS 2] Global Session: {ses_id}, expected total segments={total_seg_ses}")
            for file in os.listdir(session_path):  # loop through eye open/closed files
                if '.edf' in file:
                    file_path = os.path.join(session_path, file)
                    raw = mne.io.read_raw_edf(file_path, preload=True)
                    raw.pick(standard_channels)
                    raw = data_preprocessing(
                        raw=raw,
                        notch_freq=50,  # data collected in German, so notch at 50 Hz
                        l_freq=0.5,
                        h_freq=45.0,
                        do_bad_interp=True,
                        verbose=True
                    )
                    original_fs = raw.info['sfreq']
                    data_raw = raw.get_data().T.astype('float32')  # (C, T) -> (T, C)
                    total_seconds_all += data_raw.shape[0] / original_fs
                    print("raw shape:", data_raw.shape)
                    T_raw = data_raw.shape[0]
                    for fs in SAMPLE_RATE_LIST:
                        # resample to target fs
                        data = resample_time_series(data_raw, original_fs, fs)
                        T_res, _ = data.shape
                        print(f"fs={fs}Hz: resampled shape={data.shape}")
            
                        # overlapped sliding window with fixed STEP (in timestamps)
                        starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                        print(f"fs={fs}Hz: segments={len(starts)}")
            
                        for s in starts:
                            if cur >= N_total:
                                raise RuntimeError("Exceeded predicted N_total.")
            
                            X_mm[cur] = data[s:s + SEG_LEN]   # (SEG_LEN, C)
                            y_mm[cur, 0] = float(label)       # label
                            y_mm[cur, 1] = float(ses_id)      # Global session ID
                            y_mm[cur, 2] = float(fs)          # sample rate (scale)
                            cur += 1
            print("Total segments for Session", ses_id, ":", sum(session_segment_counts[ses_id][fs] for fs in SAMPLE_RATE_LIST))
            ses_id += 1
            print("----------------End of Session-----------------\n")
    print("----------------End of Subject-----------------\n")

    
total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

BACA-RS/sub-001\ses-1\eeg/
[PASS 2] Global Session: 1, expected total segments=2598
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-TMLR-Rebuttal\BACA-RS\BACA-RS\sub-001\ses-1\eeg\sub-001_ses-1_task-EyesClosed_acq-post_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 192999  =      0.000 ...   192.999 secs...
✔ Step 2, Montage set: 'standard_1005'.
✔ Step 3: Notch @ 50 Hz
✔ Step 4: Band-pass 0.5–45.0 Hz
ℹ Step 5: No bads to interpolate (set raw.info['bads'] first if needed)
✔ Step 6: Average reference
Fitting ICA to data using 64 channels (please be patient, this may take a while)
Selecting by non-zero PCA components: 62 components
Fitting ICA took 24.3s.
✔ ICA fitted.
Applying ICA to Raw instance
    Transforming to ICA space (62 components)
    Zeroing out 3 ICA components
    Projecting back using 64 PCA components
✔ ICA applied. Excluded components: [4, 39, 48]
raw shape: (193000, 64)
fs=200Hz

## Create JSON file

In [14]:
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "MONTAGE": "standard_1020",
    "CHANNELS": common_channels,
    "LABELS": {
        "disease_id": {0: "Healthy Control (HC)"},  # all subjects are healthy controls
        "subject_id": "integer from 1 to number of subjects",
        "sampling_frequency": "integer in SAMPLE_RATE_LIST"
    }
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

Saved meta: Processed\L200\BACA-RS\meta.json


## Load and check the processed data

In [15]:
# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]

print("Meta:")
for k, v in meta.items():
    print(f"{k}: {v}")
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

total_samples = 0
for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject
    total_samples += n_seg
    
    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

print("\nTotal subjects:", len(subject_ids))
print("Total samples:", total_samples)
# option: delete memmap views
del X_mm, y_mm

Meta:
N: 2128812
T: 200
C: 64
SAMPLE_RATE_LIST: [200, 100, 50]
MONTAGE: standard_1020
CHANNELS: ['FC1', 'F4', 'FC6', 'FC3', 'AF8', 'PO8', 'AF3', 'FT10', 'C5', 'F5', 'AF4', 'CP2', 'F7', 'T8', 'Pz', 'CPz', 'PO3', 'P4', 'F1', 'F2', 'POz', 'CP5', 'F8', 'T7', 'Cz', 'AF7', 'C2', 'TP9', 'TP7', 'CP3', 'TP8', 'Fz', 'P8', 'P5', 'C6', 'C4', 'FC2', 'Fp1', 'FT7', 'P7', 'CP1', 'F6', 'Oz', 'C3', 'PO9', 'P2', 'O2', 'FT9', 'PO4', 'C1', 'P6', 'TP10', 'Fp2', 'CP6', 'FC4', 'CP4', 'P3', 'P1', 'PO7', 'PO10', 'O1', 'FC5', 'F3', 'FT8']
LABELS: {'disease_id': {'0': 'Healthy Control (HC)'}, 'subject_id': 'integer from 1 to number of subjects', 'sampling_frequency': 'integer in SAMPLE_RATE_LIST'}
-------------------------------------
Subject ID 001: X shape=(2598, 200, 64), y shape=(2598, 3)
Subject ID 002: X shape=(2581, 200, 64), y shape=(2581, 3)
Subject ID 003: X shape=(2567, 200, 64), y shape=(2567, 3)
Subject ID 004: X shape=(2706, 200, 64), y shape=(2706, 3)
Subject ID 005: X shape=(2738, 200, 64), y shap